# PSA Two-Stage Removal Sensitivity (Multi-Component + Steady State)

This notebook models a two-stage PSA system for **H2, CO, CO2, H2O, N2, O2, CH4**. It supports:

- Different inlet compositions
- Different removal efficiencies per component in Stage 1 and Stage 2
- Optional recycle of Stage 2 tail gas back to Stage 1 feed
- **Steady-state** solution for recycle loops (fixed-point iteration)

**Assumptions**
- Total fresh feed basis = 1.0 (fractional flow).
- Removal efficiencies represent the fraction of each component removed to that stage's **tail gas**.
- No removal of a component unless specified (0–1 range).
- Stage tail gas contains only removed fractions of each component.
- Stage product gas contains the remaining fractions of each component.
- Recycle is a fraction of Stage 2 tail gas; the rest vents.


In [ ]:
import pandas as pd

COMPONENTS = ["H2", "CO", "CO2", "H2O", "N2", "O2", "CH4"]


def normalize(comp):
    total = sum(comp.values())
    if total <= 0:
        raise ValueError("Total composition must be > 0")
    return {k: v / total for k, v in comp.items()}


def apply_removal(stream, removal):
    # stream/removal are dicts by component
    tail = {k: stream[k] * removal[k] for k in COMPONENTS}
    prod = {k: stream[k] - tail[k] for k in COMPONENTS}
    return prod, tail


def comp_to_series(comp):
    total = sum(comp.values())
    if total == 0:
        return pd.Series(dict.fromkeys(COMPONENTS, 0) | {"Total": 0.0})
    return pd.Series({k: comp[k] / total for k in COMPONENTS} | {"Total": total})


def steady_state(fresh_feed, r1, r2, recycle_frac=0.0, tol=1e-8, max_iter=200):
    # Normalize fresh feed
    fresh = normalize(fresh_feed)

    # Initialize recycle stream
    recycle = dict.fromkeys(COMPONENTS, 0.0)

    for _ in range(max_iter):
        # Stage 1 feed = fresh + recycle
        s1_feed = {k: fresh[k] + recycle[k] for k in COMPONENTS}
        # Stage 1 removal
        s1_prod, s1_tail = apply_removal(s1_feed, r1)
        # Stage 2 feed = Stage 1 product
        s2_feed = s1_prod
        # Stage 2 removal
        s2_prod, s2_tail = apply_removal(s2_feed, r2)

        # Recycle fraction of Stage 2 tail
        new_recycle = {k: s2_tail[k] * recycle_frac for k in COMPONENTS}

        # Convergence check
        diff = sum(abs(new_recycle[k] - recycle[k]) for k in COMPONENTS)
        recycle = new_recycle
        if diff < tol:
            break

    # Compute vented (non-recycled) Stage 2 tail
    s2_tail_vent = {k: s2_tail[k] * (1.0 - recycle_frac) for k in COMPONENTS}

    # Compose summary
    streams = {
        "Fresh Feed": fresh,
        "Stage 1 Feed": s1_feed,
        "Stage 1 Product": s1_prod,
        "Stage 1 Tail": s1_tail,
        "Stage 2 Feed": s2_feed,
        "Stage 2 Product": s2_prod,
        "Stage 2 Tail (total)": s2_tail,
        "Stage 2 Tail Vent": s2_tail_vent,
        "Recycle to Stage 1": recycle,
    }

    df = pd.DataFrame({name: comp_to_series(comp) for name, comp in streams.items()}).T
    return df

## Example

- Feed: H2=0.60, CO=0.10, CO2=0.10, H2O=0.02, N2=0.15, O2=0.02, CH4=0.01
- Stage 1 removals: CO2/H2O high; N2/O2 moderate; CO moderate; H2 minimal
- Stage 2 removals: N2/O2/CO high; CO2/H2O moderate; H2 minimal
- Recycle fraction: 0.5 of Stage 2 tail


In [ ]:
fresh_feed = {
    "H2": 0.60,
    "CO": 0.10,
    "CO2": 0.10,
    "H2O": 0.02,
    "N2": 0.15,
    "O2": 0.02,
    "CH4": 0.01,
}

r1 = {
    "H2": 0.02,
    "CO": 0.30,
    "CO2": 0.80,
    "H2O": 0.85,
    "N2": 0.30,
    "O2": 0.30,
    "CH4": 0.20,
}
r2 = {
    "H2": 0.01,
    "CO": 0.80,
    "CO2": 0.50,
    "H2O": 0.50,
    "N2": 0.90,
    "O2": 0.90,
    "CH4": 0.30,
}

df = steady_state(fresh_feed, r1, r2, recycle_frac=0.5)

df.style.format(dict.fromkeys(df.columns, "{:.4f}"))

## Interactive Sliders (Simple UI)

If `ipywidgets` is available, this will show sliders. If not, install it with:

```bash
python3 -m pip install --user ipywidgets
```


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    # Feed sliders
    feed_sliders = {
        "H2": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.1, min=0.0, max=1.0, step=0.01, description="Feed CH4"
        ),
    }

    # Stage 1 removal sliders
    r1_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CH4"
        ),
    }

    # Stage 2 removal sliders
    r2_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CH4"
        ),
    }

    recycle = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.05, description="Recycle frac"
    )

    def update(*args):
        feed = {k: v.value for k, v in feed_sliders.items()}
        r1 = {k: v.value for k, v in r1_sliders.items()}
        r2 = {k: v.value for k, v in r2_sliders.items()}
        df = steady_state(feed, r1, r2, recycle_frac=recycle.value)
        display(df.style.format(dict.fromkeys(df.columns, "{:.4f}")))

    for group in [feed_sliders, r1_sliders, r2_sliders]:
        for w in group.values():
            w.observe(lambda change: update(), names="value")
    recycle.observe(lambda change: update(), names="value")

    feed_box = widgets.VBox(list(feed_sliders.values()))
    r1_box = widgets.VBox(list(r1_sliders.values()))
    r2_box = widgets.VBox(list(r2_sliders.values()))
    tabs = widgets.Tab(children=[feed_box, r1_box, r2_box])
    tabs.set_title(0, "Feed")
    tabs.set_title(1, "Stage 1 Removal")
    tabs.set_title(2, "Stage 2 Removal")
    display(recycle)
    display(tabs)
    update()

except Exception as e:
    print("ipywidgets not available or failed to load:", e)
    print("You can still run steady_state(...) manually.")

## SCFM Mode (Absolute Flow Basis)

This section lets you specify **total fresh feed flow (SCFM)** and inlet composition (%). It reports absolute SCFM for each stream (stage products and tails), plus composition.


In [ ]:
def steady_state_scfm(
    total_feed_scfm,
    feed_pct,
    r1,
    r2,
    recycle_frac=0.0,
    normalize_feed=True,
    tol=1e-8,
    max_iter=200,
):
    # Convert % to fractional composition
    feed_frac = {k: feed_pct[k] / 100.0 for k in COMPONENTS}
    # Normalize in case % do not sum to 100 (optional)
    if normalize_feed:
        feed_frac = normalize(feed_frac)

    # Convert to absolute SCFM
    fresh = {k: feed_frac[k] * total_feed_scfm for k in COMPONENTS}

    # Reuse steady-state solver logic on absolute flows
    recycle = dict.fromkeys(COMPONENTS, 0.0)

    for _ in range(max_iter):
        s1_feed = {k: fresh[k] + recycle[k] for k in COMPONENTS}
        s1_prod, s1_tail = apply_removal(s1_feed, r1)
        s2_feed = s1_prod
        s2_prod, s2_tail = apply_removal(s2_feed, r2)

        new_recycle = {k: s2_tail[k] * recycle_frac for k in COMPONENTS}
        diff = sum(abs(new_recycle[k] - recycle[k]) for k in COMPONENTS)
        recycle = new_recycle
        if diff < tol:
            break

    s2_tail_vent = {k: s2_tail[k] * (1.0 - recycle_frac) for k in COMPONENTS}

    streams = {
        "Fresh Feed": fresh,
        "Stage 1 Feed": s1_feed,
        "Stage 1 Product": s1_prod,
        "Stage 1 Tail": s1_tail,
        "Stage 2 Feed": s2_feed,
        "Stage 2 Product": s2_prod,
        "Stage 2 Tail (total)": s2_tail,
        "Stage 2 Tail Vent": s2_tail_vent,
        "Recycle to Stage 1": recycle,
    }

    # Build two tables: composition (%) and absolute SCFM
    comp_rows = {}
    flow_rows = {}
    for name, comp in streams.items():
        total = sum(comp.values())
        if total == 0:
            comp_rows[name] = dict.fromkeys(COMPONENTS, 0.0) | {"Total SCFM": 0.0}
            flow_rows[name] = dict.fromkeys(COMPONENTS, 0.0) | {"Total SCFM": 0.0}
        else:
            comp_rows[name] = {k: (comp[k] / total) * 100.0 for k in COMPONENTS} | {
                "Total SCFM": total
            }
            flow_rows[name] = {k: comp[k] for k in COMPONENTS} | {"Total SCFM": total}

    comp_df = pd.DataFrame(comp_rows).T
    flow_df = pd.DataFrame(flow_rows).T
    return comp_df, flow_df

### Example (SCFM)


In [ ]:
total_feed_scfm = 100.0
feed_pct = {"H2": 60, "CO": 10, "CO2": 10, "H2O": 2, "N2": 15, "O2": 2, "CH4": 1}

r1 = {
    "H2": 0.02,
    "CO": 0.30,
    "CO2": 0.80,
    "H2O": 0.85,
    "N2": 0.30,
    "O2": 0.30,
    "CH4": 0.20,
}
r2 = {
    "H2": 0.01,
    "CO": 0.80,
    "CO2": 0.50,
    "H2O": 0.50,
    "N2": 0.90,
    "O2": 0.90,
    "CH4": 0.30,
}

comp_df, flow_df = steady_state_scfm(
    total_feed_scfm, feed_pct, r1, r2, recycle_frac=0.5
)

print("Composition (% vol) by stream")
comp_df.style.format("{:.2f}")

In [ ]:
print("Absolute flow (SCFM) by stream")
flow_df.style.format("{:.2f}")

### Interactive SCFM UI

Sliders for total feed flow (SCFM), feed composition (%), per‑stage removals, and recycle fraction.


### Feed % Normalization

The SCFM UI includes an **Auto-normalize feed %** checkbox.
- When enabled, feed % values are normalized to 100 automatically.
- When disabled, a warning appears if the sum is not 100; calculations still normalize to keep mass balance.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    total_feed = widgets.FloatSlider(
        value=100.0, min=1.0, max=2000.0, step=1.0, description="Feed SCFM"
    )
    auto_normalize = widgets.Checkbox(value=True, description="Auto-normalize feed %")
    warn_out = widgets.Output()

    feed_pct_sliders = {
        "H2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % H2"
        ),
        "CO": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CO"
        ),
        "CO2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % H2O"
        ),
        "N2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % N2"
        ),
        "O2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % O2"
        ),
        "CH4": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CH4"
        ),
    }

    r1_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CH4"
        ),
    }

    r2_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CH4"
        ),
    }

    recycle = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.05, description="Recycle frac"
    )

    def update(*args):
        feed_pct = {k: v.value for k, v in feed_pct_sliders.items()}
        r1 = {k: v.value for k, v in r1_sliders.items()}
        r2 = {k: v.value for k, v in r2_sliders.items()}
        total_pct = sum(feed_pct.values())
        with warn_out:
            warn_out.clear_output()
            if not auto_normalize.value and abs(total_pct - 100.0) > 1e-6:
                print(
                    f"Warning: feed % sums to {total_pct:.2f}, normalizing to 100 for calculations."
                )
        comp_df, flow_df = steady_state_scfm(
            total_feed.value,
            feed_pct,
            r1,
            r2,
            recycle_frac=recycle.value,
            normalize_feed=True,
        )
        display(comp_df.style.format("{:.2f}"))
        display(flow_df.style.format("{:.2f}"))

    for group in [feed_pct_sliders, r1_sliders, r2_sliders]:
        for w in group.values():
            w.observe(lambda change: update(), names="value")
    total_feed.observe(lambda change: update(), names="value")
    recycle.observe(lambda change: update(), names="value")
    auto_normalize.observe(lambda change: update(), names="value")

    feed_box = widgets.VBox(list(feed_pct_sliders.values()))
    r1_box = widgets.VBox(list(r1_sliders.values()))
    r2_box = widgets.VBox(list(r2_sliders.values()))
    tabs = widgets.Tab(children=[feed_box, r1_box, r2_box])
    tabs.set_title(0, "Feed %")
    tabs.set_title(1, "Stage 1 Removal")
    tabs.set_title(2, "Stage 2 Removal")
    display(total_feed)
    display(auto_normalize)
    display(warn_out)
    display(recycle)
    display(tabs)
    update()

except Exception as e:
    print("ipywidgets not available or failed to load:", e)
    print("You can still run steady_state_scfm(...) manually.")

## Stream Map (per diagram)

This section aligns model streams to the diagram:

- **1**: Fresh feed
- **4**: PSA2 tail recycle to mixed feed
- **5A**: Mixed feed (fresh + recycle)
- **5B**: Compressor inlet (same as 5A in this simple model)
- **5C**: Compressor outlet (composition unchanged)
- **PSA1**: Stage 1
- **2**: Stage 1 exhaust (tail)
- **6**: Interstage (Stage 1 product to Stage 2 feed)
- **PSA2**: Stage 2
- **3G**: Gross product (Stage 2 product)
- **3R**: Product recycle (fraction of 3G)
- **3N**: Net product (3G minus 3R)


**Diagram reference**: The stream numbering here follows the attached diagram from the prompt.

If you save that image as `psa_streams.png` in this folder, you can embed it with:

```markdown
![](psa_streams.png)
```


In [ ]:
def steady_state_scfm_with_product_recycle(
    total_feed_scfm,
    feed_pct,
    r1,
    r2,
    tail_recycle_frac=0.0,
    product_recycle_frac=0.0,
    normalize_feed=True,
    tol=1e-8,
    max_iter=200,
):
    # Convert % to fractional composition
    feed_frac = {k: feed_pct[k] / 100.0 for k in COMPONENTS}
    if normalize_feed:
        feed_frac = normalize(feed_frac)

    fresh = {k: feed_frac[k] * total_feed_scfm for k in COMPONENTS}

    tail_recycle = dict.fromkeys(COMPONENTS, 0.0)
    product_recycle = dict.fromkeys(COMPONENTS, 0.0)

    for _ in range(max_iter):
        mixed_feed = {
            k: fresh[k] + tail_recycle[k] + product_recycle[k] for k in COMPONENTS
        }  # 5A/5B
        s1_prod, s1_tail = apply_removal(mixed_feed, r1)  # s1_tail = stream 2
        s2_feed = s1_prod  # stream 6
        s2_prod, s2_tail = apply_removal(s2_feed, r2)  # s2_prod = 3G, s2_tail = tail

        new_tail_recycle = {
            k: s2_tail[k] * tail_recycle_frac for k in COMPONENTS
        }  # stream 4
        new_product_recycle = {
            k: s2_prod[k] * product_recycle_frac for k in COMPONENTS
        }  # stream 3R

        diff = sum(
            abs(new_tail_recycle[k] - tail_recycle[k])
            + abs(new_product_recycle[k] - product_recycle[k])
            for k in COMPONENTS
        )
        tail_recycle = new_tail_recycle
        product_recycle = new_product_recycle
        if diff < tol:
            break

    s2_tail_vent = {k: s2_tail[k] * (1.0 - tail_recycle_frac) for k in COMPONENTS}
    net_product = {k: s2_prod[k] * (1.0 - product_recycle_frac) for k in COMPONENTS}

    streams = {
        "1 Fresh Feed": fresh,
        "4 Tail Recycle": tail_recycle,
        "5A Mixed Feed": mixed_feed,
        "5B Compressor Inlet": mixed_feed,
        "5C Compressor Outlet": mixed_feed,
        "2 Stage 1 Tail": s1_tail,
        "6 Interstage": s2_feed,
        "3G Gross Product": s2_prod,
        "3R Product Recycle": product_recycle,
        "3N Net Product": net_product,
        "Stage 2 Tail (total)": s2_tail,
        "Stage 2 Tail Vent": s2_tail_vent,
    }

    comp_rows = {}
    flow_rows = {}
    for name, comp in streams.items():
        total = sum(comp.values())
        if total == 0:
            comp_rows[name] = dict.fromkeys(COMPONENTS, 0.0) | {"Total SCFM": 0.0}
            flow_rows[name] = dict.fromkeys(COMPONENTS, 0.0) | {"Total SCFM": 0.0}
        else:
            comp_rows[name] = {k: (comp[k] / total) * 100.0 for k in COMPONENTS} | {
                "Total SCFM": total
            }
            flow_rows[name] = {k: comp[k] for k in COMPONENTS} | {"Total SCFM": total}

    comp_df = pd.DataFrame(comp_rows).T
    flow_df = pd.DataFrame(flow_rows).T
    return comp_df, flow_df

### Example (SCFM, with product recycle)


In [ ]:
total_feed_scfm = 100.0
feed_pct = {"H2": 60, "CO": 10, "CO2": 10, "H2O": 2, "N2": 15, "O2": 2, "CH4": 1}

r1 = {
    "H2": 0.02,
    "CO": 0.30,
    "CO2": 0.80,
    "H2O": 0.85,
    "N2": 0.30,
    "O2": 0.30,
    "CH4": 0.20,
}
r2 = {
    "H2": 0.01,
    "CO": 0.80,
    "CO2": 0.50,
    "H2O": 0.50,
    "N2": 0.90,
    "O2": 0.90,
    "CH4": 0.30,
}

comp_df, flow_df = steady_state_scfm_with_product_recycle(
    total_feed_scfm, feed_pct, r1, r2, tail_recycle_frac=0.5, product_recycle_frac=0.1
)

print("Composition (% vol) by stream")
comp_df.style.format("{:.2f}")

In [ ]:
print("Absolute flow (SCFM) by stream")
flow_df.style.format("{:.2f}")

### Interactive SCFM UI (Tail + Product Recycle)


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    total_feed = widgets.FloatSlider(
        value=100.0, min=1.0, max=2000.0, step=1.0, description="Feed SCFM"
    )
    auto_normalize = widgets.Checkbox(value=True, description="Auto-normalize feed %")
    warn_out = widgets.Output()

    feed_pct_sliders = {
        "H2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % H2"
        ),
        "CO": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CO"
        ),
        "CO2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % H2O"
        ),
        "N2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % N2"
        ),
        "O2": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % O2"
        ),
        "CH4": widgets.FloatSlider(
            value=10.0, min=0.0, max=100.0, step=0.5, description="Feed % CH4"
        ),
    }

    r1_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S1 rem CH4"
        ),
    }

    r2_sliders = {
        "H2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2"
        ),
        "CO": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO"
        ),
        "CO2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CO2"
        ),
        "H2O": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem H2O"
        ),
        "N2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem N2"
        ),
        "O2": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem O2"
        ),
        "CH4": widgets.FloatSlider(
            value=0.5, min=0.0, max=1.0, step=0.01, description="S2 rem CH4"
        ),
    }

    tail_recycle = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.05, description="Tail recycle"
    )
    product_recycle = widgets.FloatSlider(
        value=0.0, min=0.0, max=1.0, step=0.05, description="Product recycle"
    )

    def update(*args):
        feed_pct = {k: v.value for k, v in feed_pct_sliders.items()}
        r1 = {k: v.value for k, v in r1_sliders.items()}
        r2 = {k: v.value for k, v in r2_sliders.items()}
        total_pct = sum(feed_pct.values())
        with warn_out:
            warn_out.clear_output()
            if not auto_normalize.value and abs(total_pct - 100.0) > 1e-6:
                print(
                    f"Warning: feed % sums to {total_pct:.2f}, normalizing to 100 for calculations."
                )
        comp_df, flow_df = steady_state_scfm_with_product_recycle(
            total_feed.value,
            feed_pct,
            r1,
            r2,
            tail_recycle_frac=tail_recycle.value,
            product_recycle_frac=product_recycle.value,
            normalize_feed=True,
        )
        display(comp_df.style.format("{:.2f}"))
        display(flow_df.style.format("{:.2f}"))

    for group in [feed_pct_sliders, r1_sliders, r2_sliders]:
        for w in group.values():
            w.observe(lambda change: update(), names="value")
    total_feed.observe(lambda change: update(), names="value")
    auto_normalize.observe(lambda change: update(), names="value")
    tail_recycle.observe(lambda change: update(), names="value")
    product_recycle.observe(lambda change: update(), names="value")

    feed_box = widgets.VBox(list(feed_pct_sliders.values()))
    r1_box = widgets.VBox(list(r1_sliders.values()))
    r2_box = widgets.VBox(list(r2_sliders.values()))
    tabs = widgets.Tab(children=[feed_box, r1_box, r2_box])
    tabs.set_title(0, "Feed %")
    tabs.set_title(1, "Stage 1 Removal")
    tabs.set_title(2, "Stage 2 Removal")
    display(total_feed)
    display(auto_normalize)
    display(warn_out)
    display(tail_recycle)
    display(product_recycle)
    display(tabs)
    update()

except Exception as e:
    print("ipywidgets not available or failed to load:", e)
    print("You can still run steady_state_scfm_with_product_recycle(...) manually.")